In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Users\mutun\Downloads\telco_customer_churn (1).csv")

In [3]:
df.shape

(7043, 21)

In [6]:
#Inspecting data types
df.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

In [8]:
#Cleaning TotalCharges to the correct data type
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors = 'coerce')

In [9]:
#Checking for missing values
df.isna().sum()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

In [10]:
#Handling missing values
#Since tenure is 0 for these rows , setting TotalCharges to 0.0 is accurate
df['TotalCharges'] = df['TotalCharges'].fillna(0.0)

In [16]:
#Converting Churn into binary numeric column(1 = Yes, 0 = No) , this makes calculating churn rates and averages much easier
df['Churn_Flag'] = df['Churn'].map({'Yes': 1, 'No': 0})

In [17]:
print("\nCleaned Dataset Summary:")
print(df[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn', 'Churn_Flag']].head())
print("\nAny remaining missing values:", df.isna().sum().sum())


Cleaned Dataset Summary:
   customerID  tenure  MonthlyCharges  TotalCharges Churn  Churn_Flag
0  7590-VHVEG       1           29.85         29.85    No           0
1  5575-GNVDE      34           56.95       1889.50    No           0
2  3668-QPYBK       2           53.85        108.15   Yes           1
3  7795-CFOCW      45           42.30       1840.75    No           0
4  9237-HQITU       2           70.70        151.65   Yes           1

Any remaining missing values: 0


In [18]:
total_customers = len(df)

In [19]:
churned_customers = df['Churn_Flag'].sum()

In [20]:
retained_customers = total_customers - churned_customers

In [21]:
baseline_churn_rate = (churned_customers / total_customers) * 100

In [22]:
baseline_retention_rate = 100 - baseline_churn_rate

In [23]:
print("--- BASELINE CHURN RATE METRICS ---")
print(f"Total Customers: {total_customers:,}")
print(f"Churned Customers: {churned_customers:,}")
print(f"Retained Customers: {retained_customers:,}")
print(f"Baseline Churn Rate: {baseline_churn_rate:.2f}%")
print(f"Retention Rate: {baseline_retention_rate:.2f}%")

--- BASELINE CHURN RATE METRICS ---
Total Customers: 7,043
Churned Customers: 1,869
Retained Customers: 5,174
Baseline Churn Rate: 26.54%
Retention Rate: 73.46%


In [24]:
#Create Tenure Cohorts (Subscription Vintage)
bins = [-1, 12, 24, 48, 72]

In [25]:
labels = ['0 - 12 Months', '13-14 Months', '25 - 48 Months', '49+ Months']

In [26]:
df['Tenure_Cohort'] = pd.cut(df['tenure'], bins = bins, labels = labels)

In [30]:
#Creating Monthly Cgarges Tiers
charge_bins = [0, 35, 70, 120]

In [31]:
charge_labels = ['Low ($0 - $35)', 'Medium ($36 - $70)', 'High ($71+)']

In [33]:
df['Charge_Tier'] = pd.cut(df['MonthlyCharges'], bins = charge_bins, labels = charge_labels)

In [34]:
#Generating Cohort chun matrix (Tenure vs Contract Type)
churn_matrix = df.pivot_table(
    index = 'Tenure_Cohort',
    columns = 'Contract',
    values = 'Churn_Flag',
    aggfunc = 'mean',
    observed = False
) * 100

In [35]:
print("Churn Rate (%) by Tenure Cohort and Contract Type:")
print(churn_matrix.round(2))

Churn Rate (%) by Tenure Cohort and Contract Type:
Contract        Month-to-month  One year  Two year
Tenure_Cohort                                     
0 - 12 Months            51.35     10.48      0.00
13-14 Months             37.72      8.12      0.00
25 - 48 Months           32.92     10.62      2.19
49+ Months               26.02     12.93      3.33


In [36]:
#Exporting enriched dataset for Power BI
df.to_csv('Telco_Churn_Segmented.csv', index = False)